# Unit 07｜神经网络与图代理模型的位置

## Goal

在同一表格候选池上比较 RF ensemble 与 MLP ensemble 的均值、分歧和 UCB 选择。

本 Notebook 是确定性的人工教学实验，不是学习者已完成的研究，
也不是下游任务实验结果。


## Setup

本 Notebook 不训练 GNN、PBNN 或 DKL；它演示所有代理模型都应满足的统一输出接口。


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_all, oracle_values = make_regression(
    n_samples=120,
    n_features=8,
    n_informative=6,
    noise=8.0,
    random_state=2026,
)
candidate_ids = np.array([f"N{i:03d}" for i in range(len(X_all))])
split_rng = np.random.default_rng(2026)
labeled_indices = np.sort(
    split_rng.choice(len(X_all), size=32, replace=False)
)
pool_indices = np.setdiff1d(np.arange(len(X_all)), labeled_indices)
X_labeled = X_all[labeled_indices]
y_labeled = oracle_values[labeled_indices]
X_pool = X_all[pool_indices]
print("labeled/pool/features:", X_labeled.shape, X_pool.shape)


labeled/pool/features: (32, 8) (88, 8)


## Steps

按顺序执行。每个变量第一次出现时，先确认它的类型、形状和标签权限。


### 1. 建立 RF 和 MLP 成员

两者使用相同样本和目标；MLP 只在已标注集拟合 X/y 标准化，并显式检查是否达到迭代上限。


In [2]:
seeds = [11, 22, 33, 44, 55]
rf_members = [
    RandomForestRegressor(
        n_estimators=100,
        max_features="sqrt",
        random_state=seed,
        n_jobs=1,
    )
    for seed in seeds
]
mlp_members = [
    make_pipeline(
        StandardScaler(),
        TransformedTargetRegressor(
            regressor=MLPRegressor(
                hidden_layer_sizes=(24,),
                solver="lbfgs",
                alpha=0.01,
                max_iter=2000,
                tol=1e-6,
                random_state=seed,
            ),
            transformer=StandardScaler(),
        ),
    )
    for seed in seeds
]


### 2. 使用统一 fit/predict 接口

函数不接收候选标签，输出形状为成员数 × 候选数。


In [3]:
def ensemble_predictions(members):
    outputs = []
    for member in members:
        member.fit(X_labeled, y_labeled)
        outputs.append(member.predict(X_pool))
    return np.vstack(outputs)

rf_matrix = ensemble_predictions(rf_members)
mlp_matrix = ensemble_predictions(mlp_members)
mlp_iterations = [
    member.named_steps[
        "transformedtargetregressor"
    ].regressor_.n_iter_
    for member in mlp_members
]
print("RF/MLP matrices:", rf_matrix.shape, mlp_matrix.shape)
print("MLP 迭代次数:", mlp_iterations)


RF/MLP matrices: (5, 88) (5, 88)
MLP 迭代次数: [291, 453, 361, 312, 621]


### 3. 使用同一 UCB 公式

两个集成的分歧未经过概率校准，只作为教学采集信号。


In [4]:
beta = 1.0
rf_mean = rf_matrix.mean(axis=0)
rf_std = rf_matrix.std(axis=0)
mlp_mean = mlp_matrix.mean(axis=0)
mlp_std = mlp_matrix.std(axis=0)

rf_ucb = rf_mean + beta * rf_std
mlp_ucb = mlp_mean + beta * mlp_std

def select_with_tie_break(score, ids):
    order = np.lexsort((ids.astype(str), -np.asarray(score)))
    return int(order[0])

rf_query_local = select_with_tie_break(
    rf_ucb,
    candidate_ids[pool_indices],
)
mlp_query_local = select_with_tie_break(
    mlp_ucb,
    candidate_ids[pool_indices],
)
rf_query_global = int(pool_indices[rf_query_local])
mlp_query_global = int(pool_indices[mlp_query_local])


### 4. query 后查看离线结果并映射模块

一次 query 不能证明哪类模型更优。


In [5]:
# 标签揭示线
comparison = pd.DataFrame([
    {
        "surrogate": "RF ensemble",
        "candidate_id": candidate_ids[rf_query_global],
        "prediction_mean": rf_mean[rf_query_local],
        "disagreement": rf_std[rf_query_local],
        "offline_observed_y": oracle_values[rf_query_global],
    },
    {
        "surrogate": "MLP ensemble",
        "candidate_id": candidate_ids[mlp_query_global],
        "prediction_mean": mlp_mean[mlp_query_local],
        "disagreement": mlp_std[mlp_query_local],
        "offline_observed_y": oracle_values[mlp_query_global],
    },
])
roles = pd.DataFrame([
    ["RF/MLP/GNN", "表示或确定性代理"],
    ["GP/PBNN", "概率代理模型"],
    ["ensemble disagreement / posterior std", "不确定性信号"],
    ["UCB/EI", "采集函数"],
    ["查表/DFT/实验", "Oracle"],
], columns=["method", "active_learning_role"])
result_note = (
    "两类代理选中同一候选；本轮没有模型选择证据。"
    if rf_query_global == mlp_query_global
    else "两类代理选中不同候选；单轮差异不能证明谁更优。"
)
print(comparison.round(3).to_string(index=False))
print(roles.to_string(index=False))
print(result_note)


   surrogate candidate_id  prediction_mean  disagreement  offline_observed_y
 RF ensemble         N050          182.560         5.308              440.01
MLP ensemble         N050          408.323        10.483              440.01
                               method active_learning_role
                           RF/MLP/GNN             表示或确定性代理
                              GP/PBNN               概率代理模型
ensemble disagreement / posterior std               不确定性信号
                               UCB/EI                 采集函数
                            查表/DFT/实验               Oracle
两类代理选中同一候选；本轮没有模型选择证据。


## Checks

这些断言检查形状、预算和无重复等机械条件；通过断言不代表研究结论已经成立。


In [6]:
assert rf_matrix.shape == (5, len(pool_indices))
assert mlp_matrix.shape == (5, len(pool_indices))
assert max(mlp_iterations) < 2000
assert rf_query_global in pool_indices
assert mlp_query_global in pool_indices
assert set(roles["active_learning_role"]) == {
    "表示或确定性代理",
    "概率代理模型",
    "不确定性信号",
    "采集函数",
    "Oracle",
}
assert "不能证明" in result_note or "没有模型选择证据" in result_note
print("Unit 07 checks passed.")


Unit 07 checks passed.


## Next Steps

完成模型角色练习。若要运行 GNN 扩展，先完成 Day 29–35；Unit 08 将加入物理均值和人工审核。
